In [ ]:
from torch.nn import *
from IPython.display import clear_output
import torch
import numpy as np
import matplotlib.pyplot as plt
class Res(Sequential):
    def forward(s, _):
        return super().forward(_) + _
class Add(Sequential):
    def forward(s, _):
        return sum(s(_)for s in s)
class Mul(Sequential):
    def forward(s, _):
        return s[0](_) * s[1](_)
class Mt(Sequential):
    def forward(s, x):
        return super().forward(x.mT).mT
class Fn(Module):
    def __init__(s, f, *_, **__):
        super().__init__()
        s.f = f
        s._ = _
        s.__ = __
    def forward(s, _):
        return s.f(*s._, _, **s.__)
class Attn(Linear):
    def __init__(s, i, r=4, g=1):
        super().__init__(i, i*g)
        s.r = r
    def forward(s, x):
        X = functional.pad(x,(0,0,(s.r-1)//2,s.r//2))
        X = X.as_strided_((*x.shape,s.r),(*X.stride(),X.stride(-2)))
        return ((super().forward(x).unflatten(-1,(-1,x.size(-1))) @ X).softmax(-1) @ X.mT).flatten(-2)
class RoPE(Linear):
    c = {}
    def __init__(s, i):
        super().__init__(i, i, 0)
    def forward(s, x):
        c = s.c.get(x.size(-2))
        if c is None:
            c = torch.arange(x.size(-2),dtype=x.dtype,device=x.device)[:,None]*torch.logspace(0,-4,x.size(-1)//2,dtype=x.dtype,device=x.device)
            c = torch.cat((c.sin(), c.cos_()), 1)
            s.c[x.size(-2)] = c
        return c @ s.weight.T + x
def SwiGLU(i, h):
    return Mul(
        Sequential(
            Linear(i, h),
            SiLU(),
        ),
        Linear(i, h),
    )
def Model(i, h, r, R, *_):
    return Sequential(
        Res(
            Add(
                *[Attn(i, R)] if R else [],
                *[Sequential(
                    Linear(i, i, 0),
                    Mt(Conv1d(i, i, r, padding=r//2, groups=i, bias=0)),
                    *[Linear(i, i, 0)] if R else [],
                )] if r else [],
            ),
            LayerNorm(i, elementwise_affine=0, bias=0),
            SwiGLU(i, h),
            Linear(h, i, 0),
        ),
        *(
            Res(
                Linear(i, h, 0),
                Mt(MaxPool1d(2)),
                Model(h, *_),
                Linear(h, h, 0),
                Mt(ConvTranspose1d(h, h, 2, 2, groups=h, bias=0)),
                Linear(h, i, 0),
            ),
            Res(
                Add(
                    *[Attn(i, R)] if R else [],
                    *[Sequential(
                        Linear(i, i, 0),
                        Mt(Conv1d(i, i, r, padding=r//2, groups=i, bias=0)),
                        *[Linear(i, i, 0)] if R else [],
                    )] if r else [],
                ),
                LayerNorm(i, elementwise_affine=0, bias=0),
                SwiGLU(i, h),
                Linear(h, i, 0),
            ),
        ) if _ else (),
    )
dat = torch.from_numpy(np.memmap('py.bin', mode='r'))
dat = dat.as_strided_((len(dat)//512-1,1024),(512,1))
model = 16
model = Sequential(
    Embedding(92, model),
    RoPE(model),
    torch.compile(
        Model(model, model,7,8, model,7,8, model,7,8, model,7,8),
        fullgraph=True,
        dynamic=False
    ),
    Linear(model, 92),
).to('cuda',torch.bfloat16)
opt = torch.optim.Adam(model.parameters(), 1/1024)

In [ ]:
L,L0,A,A0 = [],[],[],[]
for y in range(500):
    for y in range(1):
        lc = l0c = 0
        for y in range(20):
            for y in range(20):
                t = dat[torch.randint(len(dat),[64])].to('cuda',torch.long)
                x = t.put(torch.randint(t.numel(),[t.numel()//2],device='cuda'),torch.randint(92,[t.numel()//2],device='cuda'))
                y0 = model[:-1](x)
                y = model[-1](y0)
                y0 = model[2:](y0)
                l = functional.cross_entropy(y.mT, t)
                l0 = functional.cross_entropy(y0.mT, t)
                ((l+l0)*65536).backward()
                opt.step()
                opt.zero_grad()
            lc += l.data
            l0c += l0.data
        L.append(lc[None])
        L0.append(l0c[None])
        A.append((y.argmax(-1)==t).sum()[None]/t.numel())
        A0.append((y0.argmax(-1)==t).sum()[None]/t.numel())
    plt.subplot(211).plot((L:=[torch.cat(L)])[0].to('cpu',torch.float))
    plt.plot((L0:=[torch.cat(L0)])[0].to('cpu',torch.float))
    plt.subplot(212).plot((A:=[torch.cat(A)])[0].to('cpu',torch.float))
    plt.plot((A0:=[torch.cat(A0)])[0].to('cpu',torch.float))
    clear_output(True)
    plt.show()